# Notebook 02 — Pré-processamento

## Tech Challenge Fase 1 — Classificação de SRAG (SIVEP-Gripe)

**Objetivo:** Transformar os dados brutos em um dataset limpo, codificado e pronto para modelagem.

---

## Estrutura do notebook
1. Configuração
2. Carregamento dos dados brutos
3. Filtragem de registros válidos
4. Criação do target binário
5. Seleção de features
6. Tratamento de valores ausentes
7. Codificação de variáveis categóricas
8. Análise de correlação
9. Normalização e split treino/validação/teste
10. Salvamento dos artefatos

## 1. Configuração

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.load_data import carregar_dataset
from src.preprocessing import (
    filtrar_registros_validos,
    criar_target_binario,
    selecionar_features,
    tratar_valores_ausentes,
    codificar_categoricas,
    normalizar_numericas,
    dividir_dados,
    executar_pipeline_preprocessamento,
    FEATURES, TARGET_BINARIO,
)

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
print('Configurado.')

## 2. Carregamento dos Dados Brutos

In [ ]:
df_raw = carregar_dataset()
print(f'Shape bruto: {df_raw.shape}')

## 3. Filtragem de Registros Válidos

Mantemos apenas registros com EVOLUCAO = 1 (Cura) ou 2 (Óbito).  
Removemos: Ignorados (9), Óbito por outras causas (3) e ausentes.

In [ ]:
df = filtrar_registros_validos(df_raw)
print(f'Shape após filtragem: {df.shape}')

## 4. Criação do Target Binário

Criamos a coluna `OBITO`:
- **0** → Cura (EVOLUCAO = 1)
- **1** → Óbito (EVOLUCAO = 2)

In [ ]:
df = criar_target_binario(df)

# Visualizar balanceamento
contagens = df[TARGET_BINARIO].value_counts()
print(f'\nCura (0): {contagens[0]:,} | Óbito (1): {contagens[1]:,}')
print(f'Ratio desbalanceamento: {contagens[0]/contagens[1]:.2f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Cura (0)', 'Óbito (1)'], contagens.values, color=['#2196F3', '#F44336'])
ax.set_title('Distribuição do Target Binário (OBITO)')
ax.set_ylabel('Contagem')
for i, v in enumerate(contagens.values):
    ax.text(i, v + 100, f'{v:,}\n({v/sum(contagens)*100:.1f}%)', ha='center')
plt.tight_layout()
plt.savefig('../results/figures/target_binario.png', dpi=150)
plt.show()

## 5. Seleção de Features

Selecionamos features clínicas relevantes agrupadas em:
- **Demográficas:** sexo, idade, raça, escolaridade, zona
- **Sintomas:** febre, tosse, dispneia, saturação, etc.
- **Comorbidades:** cardiopatia, diabetes, obesidade, etc.
- **Internação:** UTI, suporte ventilatório
- **Vacinação:** vacina influenza, COVID-19

In [ ]:
df_features = selecionar_features(df)
print(f'Features selecionadas: {df_features.shape[1] - 1} variáveis + 1 target')
print(f'\nColunas:')
for col in df_features.columns:
    print(f'  - {col}: {df_features[col].dtype}')

## 6. Tratamento de Valores Ausentes

**Estratégia:**
- Variáveis numéricas → imputação pela **mediana** (robusta a outliers)
- Variáveis categóricas → imputação pela **moda** (valor mais frequente)

In [ ]:
# Antes
nulos_antes = df_features.isnull().sum()
print('Nulos antes da imputação:')
print(nulos_antes[nulos_antes > 0].to_string())

df_limpo = tratar_valores_ausentes(df_features)

# Depois
nulos_depois = df_limpo.isnull().sum().sum()
print(f'\nTotal de valores nulos após imputação: {nulos_depois}')

## 7. Codificação de Variáveis Categóricas

A coluna `CS_SEXO` (M/F/I) é categórica do tipo string. Usamos **LabelEncoder** para convertê-la em numérico.  
As demais features já são numéricas no dataset SIVEP (1/2/9).

In [ ]:
df_codificado, encoders = codificar_categoricas(df_limpo)
print(f'\nDataset codificado: {df_codificado.shape}')
df_codificado.head(3)

## 8. Análise de Correlação

Verificamos a correlação entre as features e o target para:
- Identificar as variáveis mais preditivas
- Detectar multicolinearidade entre features

In [ ]:
# Correlação com o target
corr_target = df_codificado.corr()[TARGET_BINARIO].drop(TARGET_BINARIO).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
cores = ['#F44336' if v > 0 else '#2196F3' for v in corr_target.values]
ax.barh(corr_target.index[::-1], corr_target.values[::-1], color=cores[::-1])
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Correlação de Pearson com OBITO')
ax.set_title('Correlação das Features com o Target (OBITO)')
plt.tight_layout()
plt.savefig('../results/figures/correlacao_target.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features mais correlacionadas com OBITO:')
print(corr_target.head(10).to_string())

In [ ]:
# Heatmap de correlação entre features
corr_matrix = df_codificado.drop(columns=[TARGET_BINARIO]).corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3,
)
ax.set_title('Matriz de Correlação entre Features')
plt.tight_layout()
plt.savefig('../results/figures/heatmap_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Normalização e Split Treino/Validação/Teste

**Estratégia de split:** 70% treino | 15% validação | 15% teste (estratificado pelo target)

**Normalização:** StandardScaler ajustado APENAS no conjunto de treino e aplicado nos demais — evitando data leakage.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = dividir_dados(df_codificado)

print(f'\nTreino   — X: {X_train.shape}, y: {y_train.shape}')
print(f'Validação — X: {X_val.shape}, y: {y_val.shape}')
print(f'Teste    — X: {X_test.shape}, y: {y_test.shape}')

print(f'\nProporção de óbitos (treino):    {y_train.mean()*100:.1f}%')
print(f'Proporção de óbitos (validação): {y_val.mean()*100:.1f}%')
print(f'Proporção de óbitos (teste):     {y_test.mean()*100:.1f}%')

In [ ]:
X_train_norm, X_val_norm, X_test_norm, scaler = normalizar_numericas(X_train, X_val, X_test)

print('Normalização aplicada.')
print(f'\nEstatísticas de X_train após normalização:')
print(X_train_norm.describe().T[['mean', 'std']].round(3))

## 10. Salvamento dos Artefatos

Salva todos os splits e artefatos em `../data/processed/` para uso nos notebooks seguintes.

In [ ]:
# Executar pipeline completo e salvar (equivalente às etapas acima)
# Use isto para re-gerar todos os artefatos de uma vez
print('Executando pipeline completo e salvando artefatos...')
artefatos = executar_pipeline_preprocessamento(df_raw, salvar=True)
print('\nArtefatos disponíveis em data/processed/:')

processed_dir = ROOT / 'data' / 'processed'
for f in sorted(processed_dir.glob('*')):
    tamanho = f.stat().st_size / 1024
    print(f'  {f.name} ({tamanho:.0f} KB)')

## 11. Conclusões do Pré-processamento

**Resumo das decisões:**

| Etapa | Estratégia | Justificativa |
|-------|-----------|---------------|
| Filtragem | EVOLUCAO ∈ {1, 2} | Desfechos definitivos para classificação binária |
| Imputação numérica | Mediana | Robusta a outliers (ex: idades extremas) |
| Imputação categórica | Moda | Mantém a distribuição original |
| Codificação | LabelEncoder | Dataset tem variáveis ordinais (1=Sim, 2=Não, 9=Ignorado) |
| Normalização | StandardScaler | Necessário para Regressão Logística e KNN |
| Split | 70/15/15 estratificado | Mantém proporção do desbalanceamento em todos os splits |

**Próximos passos:** Notebook 03 — Modelagem